
# Multisite regression data

:func:`~uniharmony.datasets.make_multisite_regression` simulates multi-site
data with a continuous target, e.g. age in a brain-age study. Sites often
recruit different ranges of the target (a young and an old cohort): the sites
then differ in their features because of the target, not only because of the
site effect. Here the two sites overlap in the middle of the target range.


## Imports



In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

from uniharmony import verbosity
from uniharmony.combat import NeuroComBat
from uniharmony.datasets import make_multisite_regression


sns.set_theme(style="whitegrid")
verbosity("error")

## Data generation

The target is ``y = s f(X) + e`` with ``e ~ N(0, 1)``; with the default
``signal={"linear": 2.0}``, the features explain 80 % of its variance
(``R² = s² / (1 + s²)``). Site 0 has the lower 60 % of the population
targets, site 1 the upper 60 %. The site effect (a location and a scale
effect) changes the features, not the target.



In [ ]:
X, y, sites, truth = make_multisite_regression(
    n_samples=600,
    target_range_per_site=[(0, 0.6), (0.4, 1)],
    site_effect={"location": 2.0, "scale": 2.0},
    return_ground_truth=True,
    random_state=0,
)
print(f"X has {X.shape[0]} samples and {X.shape[1]} features")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(pd.DataFrame({"Target": y, "Site": sites}), x="Target", hue="Site", ax=axes[0])
axes[0].set_title("Target range of each site")
signal_direction = truth["site_free"] @ truth["target_coefficients"]  # the features combined as in the target
df = pd.DataFrame({"Target": y, "Signal direction of X (without site effect)": signal_direction, "Site": sites})
sns.scatterplot(df, x="Signal direction of X (without site effect)", y="Target", hue="Site", s=12, ax=axes[1])
axes[1].set_title("The features follow the target in both sites")
plt.tight_layout()
plt.show()

.. caution::

   The harmonizer is fitted on the whole dataset to illustrate its effect.
   In a machine-learning evaluation, fit it on the training data only.



## Harmonizing without the target removes part of it

The mean difference between the sites has two parts: the site effect and
the different targets. A harmonization that removes every difference
between the site means, such as ComBat without covariates, removes both:
the cross-validated R² of a linear model drops far below that of the data
without site effect. Applied to the data without site effect, it does the
same damage. The target signal can be kept by giving the harmonizer the
information it needs (e.g. as a covariate, with care to avoid leakage), or
by using a method designed for this case.



In [ ]:
cv = KFold(5, shuffle=True, random_state=0)
data = {
    "Without site effect (truth)": truth["site_free"],
    "Raw": X,
    "NeuroComBat, no covariates": NeuroComBat().fit_transform(X, sites=sites),
    "NeuroComBat on the data without site effect": NeuroComBat().fit_transform(truth["site_free"], sites=sites),
}
scores = {name: cross_val_score(LinearRegression(), values, y, cv=cv, scoring="r2").mean() for name, values in data.items()}
print(pd.Series(scores, name="Cross-validated R²").round(2).to_string())

.. admonition:: Take-home message

   When the sites cover different ranges of the target, the difference
   between sites is partly signal. Harmonizing it away also removes the
   signal.

